# GCP Pose Estimation — Kaggle Training

## Before running this notebook, do these 3 things on your laptop:

**1. Export your Google Drive cookies**
- Install Chrome extension **"Get cookies.txt LOCALLY"** from the Chrome Web Store
- Open a new tab → go to `drive.google.com` (logged in as `jacobantonynedu@gmail.com`)
- Click the extension icon → export → saves `cookies.txt` to Downloads

**2. Upload cookies as a private Kaggle dataset**
- kaggle.com → your profile → **Datasets** → **New Dataset**
- Upload `cookies.txt` → title: `gcp-drive-cookies` → visibility: **Private** → Create

**3. Add that dataset as input to this notebook**
- In this notebook → **Add Input** (top right) → Your Datasets → `gcp-drive-cookies` → Add

Then run cells top to bottom. GPU T4×2 + Internet must both be ON.

In [ ]:
# 1. Clone repo + install dependencies
!git clone https://github.com/J4KE-B/skylark-gcp gcp
%cd gcp
!pip -q install -r requirements.txt

In [ ]:
# 2. Download data — inject Drive auth cookies into requests so gdown uses them
# gdown's --cookies flag does not exist; we patch requests.Session instead.
import os, shutil, yaml, glob, json, http.cookiejar, requests

# Locate cookies file (added as Kaggle input dataset)
cookie_files = glob.glob('/kaggle/input/**/cookies.txt', recursive=True)
assert cookie_files, (
    "cookies.txt not found in /kaggle/input/\n"
    "Fix: Add Input -> Your Datasets -> gcp-drive-cookies"
)
cookie_file = cookie_files[0]
print(f"Found cookies: {cookie_file}")

# Load Netscape-format cookies into a CookieJar
cj = http.cookiejar.MozillaCookieJar()
cj.load(cookie_file, ignore_discard=True, ignore_expires=True)
print(f"Loaded {sum(1 for _ in cj)} cookies")

# Patch requests.Session so every session gdown creates carries our auth cookies.
# This is the correct workaround for gdown versions that don't support --cookies.
_orig_session_init = requests.Session.__init__
def _inject_cookies(self, *args, **kwargs):
    _orig_session_init(self, *args, **kwargs)
    for c in cj:
        self.cookies.set(c.name, c.value, domain=c.domain, path=c.path)
requests.Session.__init__ = _inject_cookies
print("requests.Session patched with Drive auth cookies")

# Now import and use gdown — all its internal sessions will carry our cookies
import gdown
os.makedirs('data', exist_ok=True)
print("Downloading data (10–30 min)...")
gdown.download_folder(
    id='1RDNiAO1EynKrRDomcVNXQW1-ct5zzvE5',
    output='data/',
    remaining_ok=True,
    quiet=False
)

# Restore Session to avoid side-effects in later cells
requests.Session.__init__ = _orig_session_init
print("requests.Session restored")

# gdown may have created data/<drive-folder-name>/ — flatten one level
candidates = [d for d in os.listdir('data')
              if os.path.isdir(f'data/{d}') and d not in ('train_dataset', 'test_dataset')]
if candidates:
    src = f'data/{candidates[0]}'
    print(f"Flattening {src}/ -> data/")
    for item in os.listdir(src):
        dst = f'data/{item}'
        if not os.path.exists(dst):
            shutil.move(f'{src}/{item}', dst)
    if not os.listdir(src):
        os.rmdir(src)

print("data/ contents:", sorted(os.listdir('data')))
assert os.path.exists('data/gcp_marks.json'), 'FAIL: gcp_marks.json missing'
assert os.path.isdir('data/train_dataset'),   'FAIL: train_dataset/ missing'
assert os.path.isdir('data/test_dataset'),    'FAIL: test_dataset/ missing'

# Wire config paths
cfg = yaml.safe_load(open('configs/config.yaml'))
cfg['paths'] = {'label_file': 'data/gcp_marks.json',
                'train_dir':  'data/train_dataset',
                'test_dir':   'data/test_dataset',
                'output_dir': 'outputs'}
yaml.safe_dump(cfg, open('configs/config.yaml', 'w'))

# Verify coverage
labels = json.load(open('data/gcp_marks.json'))
found  = sum(1 for k in labels if os.path.exists(f"data/train_dataset/{k}"))
pct    = 100 * found // len(labels)
print(f"Train images on disk: {found}/{len(labels)} ({pct}%)")
print("OK — enough to train." if found >= len(labels) * 0.8 else
      "WARNING: <80% images — re-export cookies (they expire) and re-run this cell.")

In [ ]:
# 3. SMOKE: 3 epochs, then inspect predictions schema before committing GPU time
import yaml
c = yaml.safe_load(open('configs/config.yaml')); c['training']['num_epochs'] = 3
yaml.safe_dump(c, open('configs/config.yaml', 'w'))
!python train.py --config configs/config.yaml
!python predict.py --config configs/config.yaml --checkpoint outputs/best.pt --output predictions.json
import json; p = json.load(open('predictions.json'))
k = next(iter(p)); print(k, '->', p[k]); print('total:', len(p))

In [ ]:
# 4. FULL run: restore 40 epochs
c = yaml.safe_load(open('configs/config.yaml')); c['training']['num_epochs'] = 40
yaml.safe_dump(c, open('configs/config.yaml', 'w'))
!python train.py --config configs/config.yaml

In [ ]:
# 5. Final predictions + download artifacts
!python predict.py --config configs/config.yaml --checkpoint outputs/best.pt --output predictions.json
import pandas as pd; pd.read_csv('outputs/training_log.csv').tail()

## Artifacts to download

From the Kaggle Output panel (right sidebar), download these three files:

- `gcp/outputs/best.pt` — model weights (upload to Drive, copy the shareable link)
- `gcp/outputs/training_log.csv` — open it, read the last row for PCK@10/25/50 + Macro-F1
- `gcp/predictions.json` — this is your submission file for Skylark

Then update `README.md` in the repo:
1. Fill in the Results table with the PCK/F1 numbers
2. Add the Drive link for `best.pt`
3. `git add README.md && git commit -m "docs: add training results and weights link" && git push`